# Tutorial 15 — Defence and concept check: the integration lab

[Lecture notes](../lectures/15_capstone.md) · [Slides](../slides/15_capstone.pptx) · [Exercise sheet](../exercises/15_capstone.md)

**40 minutes, CPU, NumPy only.** This is the lab for teams that are not presenting in the current block. You will run a small text-plus-chart system under modality ablations, score it with a paired interval, write one result record, and then add the two checks an examiner asks for: does the system use the image, and what does it do when it abstains? The twelve cases are authored and the "system" is twenty lines of deterministic code. Nothing here measures a real model.

| Minutes | Step |
|---|---|
| 0–5 | Build twelve policy-and-chart cases |
| 5–15 | Remove one input at a time |
| 15–20 | Write one result record |
| 20–40 | Your turn: two functions with tests |

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Twelve cases that need both inputs

Each case has a one-line policy ("Use plot B.") and a chart with three bars, A, B and C. The policy says which bar to read. The chart pixels give its height. Neither input alone holds the answer.

`system` reads the letter from the policy and counts dark pixels in that bar's column. With either input missing it abstains: the answer is `None` and the evidence list is empty.

In [2]:
import json

def render(values):
    im=np.full((48,72,3),255,dtype=np.uint8)
    for j,h in enumerate(values): im[40-int(h)*4:40,4+24*j:16+24*j]=[30,60,90]
    return im

def read(im,j):
    return int((im[:40,8+24*j,0]<100).sum()/4)
cases=[]
for i in range(12):
    values=rng.integers(1,10,size=3); chosen=int(rng.integers(0,3))
    cases.append({"id":f"case-{i}","split":"test","policy":f"Use plot {'ABC'[chosen]}.","image":render(values),"gold":int(values[chosen])})
def system(case,use_text=True,use_image=True):
    if not use_text or not use_image: return {"answer":None,"evidence_ids":[]}
    chosen="ABC".index(case["policy"].split()[-1][0])
    return {"answer":read(case["image"],chosen),"evidence_ids":[case["id"]+"-policy",case["id"]+"-chart"]}
print(cases[0]["policy"],system(cases[0]),"gold",cases[0]["gold"])

Use plot B. {'answer': 7, 'evidence_ids': ['case-0-policy', 'case-0-chart']} gold 7


Case 0 says "Use plot B." and the system returns 7 with two evidence IDs. A system that always answered 7 would also be right on this case. What must you change in the case to tell the two systems apart? Section 4 B does it.

## 2. Remove one input at a time

Three conditions on the same twelve cases: both inputs, policy only, chart only. Accuracy counts an abstention as wrong. Coverage is the share of cases answered. Selective accuracy is accuracy on the answered cases.

Predict all nine numbers before you run the cell.

In [3]:
conditions={"full":(True,True),"text_only":(True,False),"image_only":(False,True)}
results={}
for name,(text_on,image_on) in conditions.items():
    predictions=[system(c,text_on,image_on) for c in cases]
    correct=np.array([p["answer"]==c["gold"] for p,c in zip(predictions,cases)])
    answered=np.array([p["answer"] is not None for p in predictions])
    results[name]=correct
    print(name,{"accuracy":float(correct.mean()),"coverage":float(answered.mean()),"selective_accuracy":float(correct[answered].mean()) if answered.any() else None})
assert results["full"].all() and not results["text_only"].any()
idx=rng.integers(0,len(cases),size=(1000,len(cases)))
difference=(results["full"].astype(float)-results["image_only"].astype(float))[idx].mean(axis=1)
print("paired interval full minus image-only",np.quantile(difference,[.025,.975]))

full {'accuracy': 1.0, 'coverage': 1.0, 'selective_accuracy': 1.0}
text_only {'accuracy': 0.0, 'coverage': 0.0, 'selective_accuracy': None}
image_only {'accuracy': 0.0, 'coverage': 0.0, 'selective_accuracy': None}
paired interval full minus image-only [1. 1.]


The interval is [1, 1]. The full system is right and the chart-only system abstains on every case, so every resample gives a difference of exactly 1. The zero width comes from a deterministic toy with twelve authored cases. It says nothing about how sure you can be of a real system.

Accuracy is a mean over items, so this cell may resample the per-item differences. For macro-F1, recompute the metric on each resample, as in [notebook 14](14_paper_clinic.ipynb).

Selective accuracy is `None` when nothing was answered. Why would 0 be the wrong value to report there?

## 3. One result record

Each prediction becomes one JSON line with the fields from the [reporting templates](../TEMPLATES.md). `latency_ms` is `null` because nobody measured it. A zero would claim a measurement.

In [4]:
records=[]
for c in cases:
    p=system(c)
    row={"id":c["id"],"condition":"full","split":c["split"],"language":"en","modalities":["text","image"],"prediction":p["answer"],"target":c["gold"],"evidence_ids":p["evidence_ids"],"model":"deterministic-chart-policy-v1","revision":"course-authored-v1","prompt_version":"policy-v1","seed":42,"latency_ms":None}
    assert len(row["evidence_ids"])==2
    records.append(row)
print(json.dumps(records[0],indent=2))
# Opt-in export: uncomment after choosing an output path.
# from pathlib import Path
# Path("capstone-results.jsonl").write_text("\n".join(json.dumps(r) for r in records)+"\n")

{
  "id": "case-0",
  "condition": "full",
  "split": "test",
  "language": "en",
  "modalities": [
    "text",
    "image"
  ],
  "prediction": 7,
  "target": 7,
  "evidence_ids": [
    "case-0-policy",
    "case-0-chart"
  ],
  "model": "deterministic-chart-policy-v1",
  "revision": "course-authored-v1",
  "prompt_version": "policy-v1",
  "seed": 42,
  "latency_ms": null
}


Could a stranger rebuild your results table from records like this one? Name the field they would use to check that two conditions were scored on the same items.

## 4. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. Coverage, selective accuracy and overall accuracy.** Section 2 computes these inline. Write the function you would reuse on your own task's predictions.

In [ ]:
# YOUR TURN
def selective_scores(answers, gold):
    """Return (coverage, selective_accuracy, overall_accuracy) for a list of answers, where None means "abstained".
    Overall accuracy counts an abstention as wrong. Selective accuracy is None when nothing was answered."""
    raise NotImplementedError("your turn")

In [6]:
toy_answers = [3, 5, None, 2, 8, None, 1, 4, 9, None, 6, 7]
toy_gold    = [3, 5, 1,    2, 8, 4,    1, 4, 2, 6,    6, 5]
cov, sel, acc = selective_scores(toy_answers, toy_gold)
assert (cov, acc) == (0.75, 7 / 12) and abs(sel - 7 / 9) < 1e-12          # 9 answered, 7 of them right
assert selective_scores([None, None], [1, 2]) == (0.0, None, 0.0)
assert selective_scores([system(c)["answer"] for c in cases], [c["gold"] for c in cases]) == (1.0, 1.0, 1.0)
print(f"coverage {cov:.3f}  selective accuracy {sel:.3f}  overall accuracy {acc:.3f}")
print("A passed")

coverage 0.750  selective accuracy 0.778  overall accuracy 0.583
A passed


**B. A decoy chart.** Removing the chart makes this system abstain, which is easy to pass. A harder check keeps the policy and swaps in another case's chart, as Task 12 does with its decoy paintings. A system that uses the image should now give a different answer on most cases. A system that ignores the image will not notice.

In [ ]:
# YOUR TURN
def with_decoy(cases, shift=1):
    """Return new cases: case i keeps its id, split, policy and gold, and takes the image of case (i + shift) % len(cases).
    Do not change the input list."""
    raise NotImplementedError("your turn")

In [8]:
decoys = with_decoy(cases)
assert [d["id"] for d in decoys] == [c["id"] for c in cases] and [d["gold"] for d in decoys] == [c["gold"] for c in cases]
assert all(np.array_equal(decoys[i]["image"], cases[(i + 1) % 12]["image"]) for i in range(12))
assert all(d is not c for d, c in zip(decoys, cases)) and cases[0]["gold"] == read(cases[0]["image"], 1)   # input untouched
real = [system(c) for c in cases]; fake = [system(d) for d in decoys]
changed = sum(r["answer"] != f["answer"] for r, f in zip(real, fake))
right_only_real = sum(r["answer"] == c["gold"] and f["answer"] != c["gold"] for r, f, c in zip(real, fake, cases))
right_only_decoy = sum(r["answer"] != c["gold"] and f["answer"] == c["gold"] for r, f, c in zip(real, fake, cases))
print(f"answers changed by the decoy: {changed} of 12")
print(f"right only with the real chart: {right_only_real}   right only with the decoy: {right_only_decoy}")
print("decoy accuracy:", selective_scores([f["answer"] for f in fake], [c["gold"] for c in cases])[2],
      "  decoy answers that still cite two evidence IDs:", sum(len(f["evidence_ids"]) == 2 for f in fake), "of 12")
assert changed >= 8 and right_only_decoy == 0
print("B passed")

answers changed by the decoy: 12 of 12
right only with the real chart: 12   right only with the decoy: 0
decoy accuracy: 0.0   decoy answers that still cite two evidence IDs: 12 of 12
B passed


All twelve decoy answers are wrong, and all twelve still cite two evidence IDs. Evidence that is present has not been shown to be valid. Which check would catch a wrong answer with a correct-looking citation in your own system?

## Discussion

1. What experiment separates a system that uses a modality from one that has memorised the answer?
2. The official evaluation has not opened. What does your results table show, and what does your talk say about the test set?
3. Your main change did not beat the baseline. What makes that result worth six minutes?

## From the toy to your defence

Name the same two checks for your own task before you present. Task 12: the text-only run against the multimodal run, and the organisers' decoy-painting test. Task 6: the question without the image, and the image without the question. Task 7: a random direction of the same norm, and ablation of your steering vector. Task 9: subtask 2a against 2b, which adds the gold text frames.

Submit this notebook with both tests passing and the exercise sheet. The chart system is a teaching example and is no part of your SemEval submission.